# RIB-v2 (AEGIS adaptive gate) — train + cheap validation

**Goal:** train the input-adaptive RIB gate (#1) with clean/corrupt supervision (#2) so RIB
*backs off* on the high-baseline axes (lighting / layout / init) where the old always-on
RIB bled, while keeping the Sensor-Noise / Camera wins. Then validate on those 3 axes only.

### Manual prerequisites (do these in the right-hand panel before running)
1. **Settings → Accelerator → GPU T4 x1**, and **Internet → On**.
2. **Add Data →** attach your **720-wide base checkpoint** as a dataset. Upload the local
   folder `outputs/smolvla_spatial_repro/checkpoints/020000/pretrained_model` and make sure it
   mounts at `/kaggle/input/aegis-ckpt/pretrained_model`. (If it mounts elsewhere, fix `BASE_CKPT` in cell 4.)

Training is GPU-light (~2–4 h on T4, free). Eval is CPU-bound; the validation slice is kept tiny
so it finishes in one session.

## 1 · Install + clone

In [ ]:
import os, sys, subprocess
WORK = "/kaggle/working"; REPO = f"{WORK}/aegis"; SIB = f"{REPO}/sib_vla"
# lerobot + SmolVLA + LIBERO-Plus deps. (LIBERO/EGL on Kaggle is finicky — if a rollout
# errors later, that's the first place to debug; the TRAIN step does not need the sim.)
subprocess.run('pip -q install "lerobot[smolvla]" draccus==0.8.0 imageio imageio-ffmpeg', shell=True)
subprocess.run('pip -q install robosuite==1.4.1 bddl easydict h5py', shell=True)
if not os.path.isdir(REPO):
    subprocess.run(f'git clone --depth 1 https://github.com/A20archi/aegis.git {REPO}', shell=True)
print("repo present:", os.path.isdir(SIB))

## 2 · Patch in the adaptive-gate code (not yet on GitHub)
Applies the exact #1+#2 edits to the cloned files. Asserts each hunk lands.

In [ ]:
import io, os
SIB = "/kaggle/working/aegis/sib_vla"

def patch(path, edits):
    p = f"{SIB}/{path}"
    s = open(p).read()
    for i,(old,new) in enumerate(edits):
        if new in s and old not in s:
            print(f"  [{path}] hunk {i} already applied"); continue
        assert old in s, f"PATCH FAIL {path} hunk {i}: anchor not found"
        s = s.replace(old, new, 1)
    open(p,"w").write(s); print(f"  [{path}] patched ({len(edits)} hunks)")

# ---- robust_ib.py ----
rib_edits = [
( '''    def __init__(self, original_linear: nn.Linear, D_out: int = 960,
                 d_z: int = 448, n_heads: int = 7) -> None:
        super().__init__()
        self.linear = original_linear
        self.rib = RobustIB(D_out, d_z=d_z, n_heads=n_heads)''',
  '''    def __init__(self, original_linear: nn.Linear, D_out: int = 960,
                 d_z: int = 448, n_heads: int = 7, gate_hidden: int = 128) -> None:
        super().__init__()
        self.linear = original_linear
        self.rib = RobustIB(D_out, d_z=d_z, n_heads=n_heads)''' ),
( '''        self.fusion_coeff = nn.Parameter(torch.full((1,), 0.5493))

    def forward(self, x: Tensor) -> Tensor:
        z_mlp = self.linear(x)
        z_rib = self.rib(z_mlp)
        return z_mlp + torch.tanh(self.fusion_coeff) * z_rib''',
  '''        self.fusion_coeff = nn.Parameter(torch.full((1,), 0.5493))
        self.gate_head = nn.Sequential(nn.Linear(D_out, gate_hidden), nn.GELU(),
                                       nn.Linear(gate_hidden, 1))
        nn.init.zeros_(self.gate_head[-1].weight)
        nn.init.constant_(self.gate_head[-1].bias, 2.0)
        self.last_gate: Tensor | None = None

    def forward(self, x: Tensor) -> Tensor:
        z_mlp = self.linear(x)
        z_rib = self.rib(z_mlp)
        if z_mlp.dim() == 3:
            pooled = z_mlp.float().mean(dim=1)
            g = torch.sigmoid(self.gate_head(pooled))
            self.last_gate = g.squeeze(-1)
            g = g.unsqueeze(1)
        else:
            pooled = z_mlp.float().mean(dim=0, keepdim=True)
            g = torch.sigmoid(self.gate_head(pooled))
            self.last_gate = g.squeeze(-1)
        g = g.to(z_rib.dtype)
        return z_mlp + torch.tanh(self.fusion_coeff) * g * z_rib''' ),
( '''        "rib_state": fused.rib.state_dict(),
        "fusion_coeff": fused.fusion_coeff.data.clone(),''',
  '''        "rib_state": fused.rib.state_dict(),
        "gate_head_state": fused.gate_head.state_dict(),
        "fusion_coeff": fused.fusion_coeff.data.clone(),''' ),
( '''    fused.rib.load_state_dict(ckpt["rib_state"])
    ref = next(fused.linear.parameters())
    fused.rib.to(device=ref.device, dtype=ref.dtype)''',
  '''    fused.rib.load_state_dict(ckpt["rib_state"])
    if "gate_head_state" in ckpt:
        fused.gate_head.load_state_dict(ckpt["gate_head_state"])
    ref = next(fused.linear.parameters())
    fused.rib.to(device=ref.device, dtype=ref.dtype)
    fused.gate_head.to(device=ref.device, dtype=ref.dtype)''' ),
( '''    ref = next(original_linear.parameters())
    fused.rib.to(device=ref.device, dtype=ref.dtype)
    fused.fusion_coeff.data = fused.fusion_coeff.data.to(device=ref.device, dtype=ref.dtype)''',
  '''    ref = next(original_linear.parameters())
    fused.rib.to(device=ref.device, dtype=ref.dtype)
    fused.gate_head.to(device=ref.device, dtype=ref.dtype)
    fused.fusion_coeff.data = fused.fusion_coeff.data.to(device=ref.device, dtype=ref.dtype)''' ),
]
patch("sib/robust_ib.py", rib_edits)

# ---- finetune_rib.py ----
ft_edits = [
( '''    rib_params = list(fused.rib.parameters()) + [fused.fusion_coeff]''',
  '''    rib_params = (list(fused.rib.parameters()) + list(fused.gate_head.parameters())
                  + [fused.fusion_coeff])''' ),
( '''    ap.add_argument("--corrupt-frac", type=float, default=0.6, help="fraction of batch images corrupted")
    ap.add_argument("--tag", default="rib_on86")''',
  '''    ap.add_argument("--corrupt-frac", type=float, default=0.6, help="fraction of batch images corrupted")
    ap.add_argument("--lambda-gate", type=float, default=0.05,
                    help="weight on the adaptive-gate loss (open-on-corrupt / close-on-clean)")
    ap.add_argument("--tag", default="rib_on86")''' ),
( '''                    kl_pen = torch.clamp(kl, min=args.free_bits)   # free-bits: no pressure below floor
                    loss = task_loss + args.beta * kl_pen''',
  '''                    kl_pen = torch.clamp(kl, min=args.free_bits)   # free-bits: no pressure below floor
                    g = fused.last_gate
                    if g is not None and g.numel() == cmask.numel():
                        gate_loss = F.binary_cross_entropy(
                            g.float().clamp(1e-4, 1 - 1e-4), cmask.float())
                        gate_mode = "bce"
                    elif g is not None:
                        gate_loss = g.float().mean()
                        gate_mode = "sparse"
                    else:
                        gate_loss = torch.zeros((), device=device)
                        gate_mode = "off"
                    loss = task_loss + args.beta * kl_pen + args.lambda_gate * gate_loss''' ),
( '''                if step % 100 == 0:
                    print(f"[rib] step={step:6d}/{args.steps}  loss={loss.item():.4f}  "
                          f"task={task_loss.item():.4f}  kl={kl.item():.3f}  "
                          f"coeff={fused.ib_contribution:+.3f}  lr={opt.param_groups[0]['lr']:.1e}", flush=True)''',
  '''                if step % 100 == 0:
                    g_open = float(g.float().mean().item()) if g is not None else float("nan")
                    print(f"[rib] step={step:6d}/{args.steps}  loss={loss.item():.4f}  "
                          f"task={task_loss.item():.4f}  kl={kl.item():.3f}  "
                          f"coeff={fused.ib_contribution:+.3f}  gate[{gate_mode}]={g_open:.3f}  "
                          f"lr={opt.param_groups[0]['lr']:.1e}", flush=True)''' ),
]
patch("scripts/finetune_rib.py", ft_edits)
print("PATCH OK")

## 3 · Smoke-test the gate (CPU, ~2 s) — proves the patch took

In [ ]:
import sys; sys.path.insert(0, "/kaggle/working/aegis/sib_vla")
import torch, torch.nn as nn
from sib.robust_ib import FusedRobustIBProjector
torch.manual_seed(0); B,N,Din,D = 4,16,512,960
lin = nn.Linear(Din,D); fused = FusedRobustIBProjector(lin, D_out=D, d_z=128, n_heads=8)
x = torch.randn(B,N,Din)
with torch.no_grad(): out, ref = fused(x), lin(x)
assert torch.allclose(out, ref, atol=1e-5), "identity-at-init FAILED"
with torch.no_grad():
    nn.init.normal_(fused.rib.dec[-1].weight, std=.02); fused.gate_head[-1].bias.fill_(-30.)
    closed = (fused(x)-ref).abs().mean().item()
assert closed < 1e-5, "closed-gate != baseline"
print("gate OK: identity-at-init exact, closed-gate->baseline exact, last_gate", tuple(fused.last_gate.shape))

## 4 · Kaggle config (points at the uploaded 720-wide base + HF LIBERO data)

In [ ]:
import yaml, os
BASE_CKPT = "/kaggle/input/aegis-ckpt/pretrained_model"   # <-- fix if your dataset mounts elsewhere
assert os.path.isdir(BASE_CKPT), f"base checkpoint not found at {BASE_CKPT} — attach it as a dataset"
cfg = {
    "checkpoint": BASE_CKPT,
    "repo_id": "HuggingFaceVLA/libero",   # public LeRobot dataset; downloads on first use
    "dataset_root": None, "fps": 10, "chunk_size": 50, "n_action_steps": 1,
    "seeds": [42], "device": "cuda", "n_gpus": 1,
    "output_dir": "/kaggle/working/rib_v2",
}
os.makedirs(cfg["output_dir"], exist_ok=True)
open("/kaggle/working/kaggle_rib.yaml","w").write(yaml.safe_dump(cfg))
print(open("/kaggle/working/kaggle_rib.yaml").read())

## 5 · Train RIB-v2 (adaptive gate). Watch `gate[bce]` learn to separate clean/corrupt.

In [ ]:
import subprocess
cmd = ["python","scripts/finetune_rib.py","--config","/kaggle/working/kaggle_rib.yaml",
       "--steps","12000","--batch-size","32","--lr-rib","3e-4","--lr-head","2e-5",
       "--d-z","448","--beta","1e-3","--free-bits","0.1","--corrupt-frac","0.6",
       "--lambda-gate","0.05","--tag","rib_v2_gate"]
subprocess.run(cmd, cwd="/kaggle/working/aegis/sib_vla")
# -> writes /kaggle/working/rib_v2/rib_v2_gate.pt (checkpoints every 3k steps too)

## 6 · Cheap validation — the 3 decisive axes only, Object + Goal
baseline vs AEGIS-v2 on **Sensor Noise / Camera Viewpoints / Light Conditions**.
Go/no-go: Light should move from −5…−11 toward ~0 **without** losing the Sensor/Camera wins.

In [ ]:
import subprocess, json, os
RIB="/kaggle/working/rib_v2/rib_v2_gate.pt"
SIB="/kaggle/working/aegis/sib_vla"
BASE="/kaggle/input/aegis-ckpt/pretrained_model"
CATS=["Sensor Noise","Camera Viewpoints","Light Conditions"]
SUITES={"libero_object":280,"libero_goal":300}
OUT="/kaggle/working/val"; os.makedirs(OUT,exist_ok=True)
def run(method, suite, maxstep):
    out=f"{OUT}/{suite}_{method}.json"
    cmd=["python","scripts/libero_plus_aegis_eval.py","--method",method,"--ckpt",BASE,
         "--suite",suite,"--per-cat","6","--max-steps",str(maxstep),"--seed","42",
         "--cats",*CATS,"--out",out]
    if method=="aegis": cmd+=["--rib-weights",RIB]
    subprocess.run(cmd, cwd=SIB); return out
for su,ms in SUITES.items():
    for m in ("baseline","aegis"): run(m,su,ms)
print("validation done ->", OUT)

## 7 · Validation table

In [ ]:
import json, os
OUT="/kaggle/working/val"
for su in ["libero_object","libero_goal"]:
    b=json.load(open(f"{OUT}/{su}_baseline.json")); a=json.load(open(f"{OUT}/{su}_aegis.json"))
    print(f"\n=== {su} ===")
    for c in ["Sensor Noise","Camera Viewpoints","Light Conditions"]:
        bs=100*b["per_category"].get(c,{}).get("sr",0); as_=100*a["per_category"].get(c,{}).get("sr",0)
        print(f"  {c:20} {bs:5.1f} -> {as_:5.1f}   d={as_-bs:+5.1f}")